# 05 — Explicabilité avec Grad-CAM
> Visualisation des zones décisives du modèle.
>
> Grad-CAM (*Gradient-weighted Class Activation Mapping*) colorie en rouge/chaud les régions qui ont le plus contribué à la décision du modèle.

In [ ]:
import sys
sys.path.insert(0, '..')

# Installation si nécessaire
try:
    from pytorch_grad_cam import GradCAM
    print("✓ pytorch-grad-cam déjà installé")
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'grad-cam', '-q'])
    print("✓ pytorch-grad-cam installé")

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from config import MODEL_PATH, CLASS_NAMES
from src.model import load_model
from src.dataset import get_dataloaders
from src.gradcam import get_gradcam_heatmap, overlay_heatmap, denormalize, batch_gradcam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(MODEL_PATH, device)
_, _, test_loader, class_names = get_dataloaders(num_workers=0)
print(f"\nDevice : {device} | Modèle chargé")

## 1. Principe de Grad-CAM

1. **Forward pass** → activation de la dernière couche convolutive
2. **Backward pass** → gradients par rapport à ces activations
3. **Pondération** → moyenne globale des gradients (Global Average Pooling)
4. **Heatmap** → `ReLU(somme pondérée des feature maps)`
5. **Superposition** → redimensionnée et appliquée sur l'image originale

In [ ]:
# Démonstration sur une image du test set
images, labels = next(iter(test_loader))
img_tensor = images[0:1].to(device)
true_label  = labels[0].item()

heatmap, pred_class, confidence = get_gradcam_heatmap(model, img_tensor, device)

print(f"Vraie classe      : {class_names[true_label]}")
print(f"Classe prédite    : {class_names[pred_class]}")
print(f"Confiance         : {confidence:.1%}")
print(f"Shape heatmap     : {heatmap.shape}  (valeurs [0,1])")
print(f"  min={heatmap.min():.3f}  max={heatmap.max():.3f}  moy={heatmap.mean():.3f}")

## 2. Visualisation individuelle

In [ ]:
from src.gradcam import visualize_gradcam

# Sélection : une Benign + une Malignant
benign_imgs, malignant_imgs = [], []
for imgs, lbls in test_loader:
    for i in range(len(lbls)):
        if lbls[i].item() == 0 and len(benign_imgs) < 1:
            benign_imgs.append(imgs[i:i+1])
        if lbls[i].item() == 1 and len(malignant_imgs) < 1:
            malignant_imgs.append(imgs[i:i+1])
    if benign_imgs and malignant_imgs:
        break

for tensor, label in [(benign_imgs[0], 'Benign'), (malignant_imgs[0], 'Malignant')]:
    visualize_gradcam(model, tensor.to(device), device,
                      class_names=class_names, title=f"Exemple — {label}")

## 3. Grille complète : 6 images test

In [ ]:
batch_gradcam(model, test_loader, device,
             class_names=class_names, n_samples=6,
             save_path='../models/gradcam_grid.png')

## 4. Analyse des cas difficiles

In [ ]:
# Images mal classées avec Grad-CAM — montre où le modèle s'est trompé
from src.evaluate import get_predictions
from src.gradcam import visualize_gradcam

all_labels, all_preds, all_probs = get_predictions(model, test_loader, device)

# Collecte des erreurs
errors = []
for imgs, lbls in test_loader:
    for i in range(len(lbls)):
        true = lbls[i].item()
        pred_idx = (all_preds[len(errors)] if len(errors) < len(all_preds) else -1)
        if true != pred_idx and len(errors) < 4:
            errors.append((imgs[i:i+1], true, pred_idx))
    if len(errors) >= 4:
        break

print(f"Cas difficiles visualisés : {len(errors)}")
for tensor, true_l, pred_l in errors:
    title = f"ERREUR — Vrai:{class_names[true_l]} | Prédit:{class_names[pred_l]}"
    visualize_gradcam(model, tensor.to(device), device, class_names=class_names, title=title)

## 5. Conclusion

Complète avec tes observations :

In [ ]:
print("=" * 55)
print("ANALYSE GRAD-CAM — CONCLUSIONS")
print("=" * 55)
print("Observations à documenter dans le rapport :")
print("  1. Les zones activées correspondent-elles aux lésions ?")
print("  2. Le modèle regarde-t-il les bonnes régions ?")
print("  3. Y a-t-il des artéfacts (zones non pertinentes activées) ?")
print("  4. Les cas difficiles activent-ils des zones ambiguës ?")
print()
print("Figures sauvegardées dans models/gradcam_grid.png")
print("\n→ Ces visualisations iront dans le rapport (section Explicabilité)")